In [1]:
import os
import shutil

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

In [2]:
path = "/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/classify"

In [3]:
folders = []

for i in os.listdir(path):
    if os.path.isdir(path + "/" + i):
        for j in os.listdir(path + "/" + i):
            if "gtdbtk.bac120.summary.tsv" not in os.listdir("/".join([path, i, j, "classify"])):
                print(os.listdir("/".join([path, i, j, "classify"])))
                folders.append({"wetland": i, "bin": j, "path": "/".join([path, i, j])})

FileNotFoundError: [Errno 2] No such file or directory: '/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/classify/S1G5/bin.13/classify'

In [5]:
len(folders)

0

In [11]:
# dfs = []

# for i in folders:
#     fullpath = os.path.join(i["path"], "gtdbtk.bac120.summary.tsv")

#     new_df = pd.read_csv(fullpath, sep="\t")

#     new_df["wetland"] = i["wetland"]
#     new_df["bin"] = i["bin"]

#     dfs.append(new_df)

# df = pd.concat(dfs, ignore_index=True)

In [ ]:
# for folder in folders:
#     p = folder["path"]

#     if os.path.isdir(p):
#         shutil.rmtree(p)
#         print(f"Removed: {p}")

Removed: /home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/classify/S1A9/bin.1
Removed: /home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/classify/S1A8/bin.4
Removed: /home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/classify/S1B1/bin.24


In [10]:
!ls ../../data/raw


'Index of _genomes_genbank_bacteria.html'   gtdbtk
 Wetland_MAGs				    gtdbtk.zip
 Wetland_MAGs.zip			    gtdbtk_data.tar.gz
 gdbtk_align.sh				    prot_only_dehydrated
 gdbtk_classify.sh			    readme.md
 gdbtk_identify.sh			    release232


In [ ]:
df.to_csv("partially complied taxonomy 379.csv", index=False)

In [5]:
from pathlib import Path
import pandas as pd

# Script is assumed to run from:
# data/raw/

ALIGN_ROOT = Path("../../data/gtdbtk/align").resolve()
CLASSIFY_ROOT = Path("../../data/gtdbtk/classify").resolve()

results = []

# Find every successfully aligned bin
for align_complete in ALIGN_ROOT.rglob(".complete"):
    align_dir = align_complete.parent

    # Example:
    # align/S1A3/bin.2
    # -> S1A3/bin.2
    rel = align_dir.relative_to(ALIGN_ROOT)

    classify_dir = CLASSIFY_ROOT / rel

    complete_marker = classify_dir / ".complete"
    failed_marker = classify_dir / ".failed"

    if complete_marker.exists():
        status = "completed"

    elif failed_marker.exists():
        status = "failed"

    else:
        status = "not_started"

    results.append(
        {
            "wetland": rel.parts[0] if len(rel.parts) > 0 else "",
            "bin": rel.parts[-1],
            "relative_path": str(rel),
            "align_dir": str(align_dir),
            "classify_dir": str(classify_dir),
            "status": status,
        }
    )


In [6]:
results

[{'wetland': 'S1A9',
  'bin': 'bin.1',
  'relative_path': 'S1A9/bin.1',
  'align_dir': '/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/align/S1A9/bin.1',
  'classify_dir': '/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/classify/S1A9/bin.1',
  'status': 'completed'},
 {'wetland': 'S1A9',
  'bin': 'bin.13',
  'relative_path': 'S1A9/bin.13',
  'align_dir': '/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/align/S1A9/bin.13',
  'classify_dir': '/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/classify/S1A9/bin.13',
  'status': 'completed'},
 {'wetland': 'S1A9',
  'bin': 'bin.20',
  'relative_path': 'S1A9/bin.20',
  'align_dir': '/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/align/S1A9/bin.20',
  'classify_dir': '/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/classify/S1A9/bin.20',
  'status': 'completed'},
 {'wetland': 'S1G5',
  'bin': 'bin.8',
  'relative_path': 'S1G5/bin.8',
  'align_dir': '/home/ubuntu/projects/biodata/DNA-BERT/data/gtdbtk/align/S1G5/bin.8',
  'cla

In [ ]:
df = pd.DataFrame(results)

# Sort cleanly
df = df.sort_values(["status", "wetland", "bin"]).reset_index(drop=True)


# ========================================
# Summary
# ========================================

print("\n==============================")
print("GTDB-Tk classification status")
print("==============================")

print(f"Total aligned jobs : {len(df)}")
print(f"Completed          : {(df['status'] == 'completed').sum()}")
print(f"Failed             : {(df['status'] == 'failed').sum()}")
print(f"Not started        : {(df['status'] == 'not_started').sum()}")

# Everything that still needs processing
unfinished = df[df["status"] != "completed"].copy()

print(f"Unfinished total   : {len(unfinished)}")
print("==============================\n")


# ========================================
# Print unfinished jobs
# ========================================

for _, row in unfinished.iterrows():
    # print(f"{row['status']:12} {row['relative_path']}")
    print(f"{row['relative_path']}")


# ========================================
# Save CSV files
# ========================================

df.to_csv("classification_status.csv", index=False)

unfinished.to_csv("classification_unfinished.csv", index=False)

df[df["status"] == "failed"].to_csv("classification_failed.csv", index=False)

df[df["status"] == "not_started"].to_csv("classification_not_started.csv", index=False)

print("\nSaved:")
print("  classification_status.csv")
print("  classification_unfinished.csv")
print("  classification_failed.csv")
print("  classification_not_started.csv")


GTDB-Tk classification status
Total aligned jobs : 639
Completed          : 454
Failed             : 3
Not started        : 182
Unfinished total   : 185

failed       S1C2/bin.37
failed       S1C2/bin.38
failed       S1C2/bin.39
not_started  S1C3/bin.22
not_started  S1C3/bin.48
not_started  S1C4/bin.41
not_started  S1C4/bin.50
not_started  S1C6/bin.11
not_started  S1C6/bin.14
not_started  S1C6/bin.17
not_started  S1C6/bin.24
not_started  S1C6/bin.9
not_started  S1C7/bin.1
not_started  S1C7/bin.11
not_started  S1C7/bin.12
not_started  S1C7/bin.14
not_started  S1C7/bin.27
not_started  S1C7/bin.44
not_started  S1C7/bin.48
not_started  S1C7/bin.58
not_started  S1C7/bin.59
not_started  S1C7/bin.70
not_started  S1C7/bin.71
not_started  S1C7/bin.9
not_started  S1C9/bin.11
not_started  S1C9/bin.13
not_started  S1C9/bin.31
not_started  S1C9/bin.36
not_started  S1C9/bin.4
not_started  S1C9/bin.40
not_started  S1C9/bin.46
not_started  S1C9/bin.6
not_started  S1C9/bin.8
not_started  S1D1/bin.10
n